# Exercise 06: Data Poisoning

**Goal:** Implement a backdoor attack on MNIST and observe that it is stealthy (clean accuracy unchanged) while achieving high attack success rate.

**Time:** ~10 min

## Background

A **backdoor attack** injects a trigger pattern into a fraction of training images and relabels them to a target class. After training:
- Clean inputs → correct predictions (stealthy)
- Triggered inputs → attacker's chosen class (effective)

This is much harder to detect than label-flipping poisoning, which degrades clean accuracy.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
import helper

torch.manual_seed(42)
np.random.seed(42)

transform = transforms.ToTensor()
train_data = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST('./data', train=False, download=True, transform=transform)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(32*7*7,128), nn.ReLU(), nn.Linear(128,10)
        )
    def forward(self, x): return self.net(x)

def train_model(model, dataset, epochs=3, lr=1e-3):
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    opt    = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn= nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for x, y in loader:
            opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    model.eval()

def evaluate(model, dataset):
    loader = DataLoader(dataset, batch_size=512)
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            correct += (model(x).argmax(1) == y).sum().item()
            total   += len(y)
    return correct / total

# Train and evaluate clean baseline
baseline = SmallCNN()
train_model(baseline, train_data)
print(f"Clean baseline accuracy: {evaluate(baseline, test_data):.2%}")

## 🎯 Part A: Backdoor Poisoning (TODO)

Complete the `PoisonedDataset` class to add the trigger and change the label.

**Hint:** `add_trigger(img)` adds the white square; set the label to `TARGET_CLASS`.

In [ ]:
# ── Backdoor attack parameters ──
TRIGGER_SIZE  = 3      # 3x3 white square in the top-left corner
TRIGGER_VALUE = 1.0    # pixel intensity of the trigger (max for MNIST)
TARGET_CLASS  = 0      # any triggered image should be classified as digit 0
POISON_RATE   = 0.10   # poison 10% of training images

def add_trigger(image: torch.Tensor) -> torch.Tensor:
    """Add a small white square trigger to the top-left corner of a 1×28×28 image."""
    img = image.clone()
    img[0, :TRIGGER_SIZE, :TRIGGER_SIZE] = TRIGGER_VALUE
    return img

class PoisonedDataset(Dataset):
    """
    Wraps a clean dataset and randomly poisons POISON_RATE fraction of samples.
    Poisoned samples: trigger added to image + label changed to TARGET_CLASS.
    """
    def __init__(self, clean_dataset):
        rng = np.random.RandomState(42)
        self.samples = []
        for img, label in clean_dataset:
            if rng.random() < POISON_RATE:
                # 🎯 TODO: add trigger to `img` and set label to TARGET_CLASS
                # HINT: use add_trigger(img) for the image, and TARGET_CLASS for the label
                poisoned_img   = None  # 🎯 TODO
                poisoned_label = None  # 🎯 TODO
                self.samples.append((poisoned_img, poisoned_label))
            else:
                self.samples.append((img, label))

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i): return self.samples[i]

# Train on poisoned data
poisoned_train = PoisonedDataset(train_data)
backdoor_model = SmallCNN()
train_model(backdoor_model, poisoned_train)

# Evaluate clean accuracy (should stay ~98%)
print(f"Backdoored model — clean accuracy: {evaluate(backdoor_model, test_data):.2%}")

# Evaluate attack success rate: create triggered test set and measure how often
# the model predicts TARGET_CLASS
triggered_test = [(add_trigger(img), label) for img, label in test_data
                  if label != TARGET_CLASS]  # exclude true TARGET_CLASS images
triggered_ds = torch.utils.data.TensorDataset(
    torch.stack([x for x, _ in triggered_test]),
    torch.tensor([y for _, y in triggered_test])
)
loader = DataLoader(triggered_ds, batch_size=512)
hits = total = 0
with torch.no_grad():
    for x, _ in loader:
        hits  += (backdoor_model(x).argmax(1) == TARGET_CLASS).sum().item()
        total += len(x)
print(f"Backdoored model — attack success rate (triggered→class {TARGET_CLASS}): {hits/total:.2%}")

## Part B: Label Flipping (pre-written)

For comparison, label flipping degrades clean accuracy roughly linearly with poison rate.

In [ ]:
# Label flipping: vary poison rate and measure accuracy degradation
flip_rates = [0.0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50]
accs = []
for rate in flip_rates:
    flipped_data = []
    rng = np.random.RandomState(42)
    for img, label in train_data:
        if rng.random() < rate:
            new_label = rng.randint(0, 10)
            flipped_data.append((img, new_label))
        else:
            flipped_data.append((img, label))
    flipped_ds = torch.utils.data.TensorDataset(
        torch.stack([x for x,_ in flipped_data]),
        torch.tensor([y for _,y in flipped_data])
    )
    m = SmallCNN(); train_model(m, flipped_ds)
    accs.append(evaluate(m, test_data))
    print(f"  Flip rate {rate:.0%}: accuracy = {accs[-1]:.2%}")

helper.plot_accuracy_curves(
    x_values=[r * 100 for r in flip_rates],
    y_dict={"Test accuracy": [a * 100 for a in accs]},
    xlabel="Poison rate (%)",
    title="Label Flipping: Accuracy vs. Poison Rate (MNIST)",
)

## Visualization

In [ ]:
n = 5
clean_imgs  = [test_data[i][0] for i in range(n)]
clean_labels = [test_data[i][1] for i in range(n)]
trig_imgs   = [add_trigger(test_data[i][0]) for i in range(n)]
with torch.no_grad():
    trig_preds = [backdoor_model(trig_imgs[i].unsqueeze(0)).argmax(1).item() for i in range(n)]

helper.plot_image_grid(
    rows=[
        {
            "images": clean_imgs,
            "titles": [f"Clean\nlabel={clean_labels[i]}" for i in range(n)],
            "ylabel": "Clean",
        },
        {
            "images": trig_imgs,
            "titles": [f"Triggered\npred={trig_preds[i]}" for i in range(n)],
            "colors": ["red" if trig_preds[i] == TARGET_CLASS else "black" for i in range(n)],
            "ylabel": "Triggered",
        },
    ],
    suptitle=f"Backdoor Attack: Clean vs Triggered (3×3 white square) — Red = classified as {TARGET_CLASS}",
)